<a href="https://colab.research.google.com/github/Shadoww002/NLP/blob/main/Phase-4/IMDB_movies_review_sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [54]:
import numpy as np
import pandas as pd
df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [55]:
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [56]:
df.isnull().sum()

,0
review,0
sentiment,0


In [57]:
df.duplicated().sum()

np.int64(418)

In [58]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [59]:
import re
def remove_tags(raw_text):
  cleaned_text = re.sub(re.compile("<.*?>"),"",raw_text)
  return cleaned_text

In [60]:
df["review"] = df["review"].apply(remove_tags)

In [61]:
df["review"] = df["review"].apply(lambda x:x.lower())

In [62]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()   # simple tokenization
    filtered = [w for w in words if w not in stop_words]
    return " ".join(filtered)

df['clean_review'] = df['review'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [63]:
df.head()

,review,sentiment,clean_review
0,one of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching 1 oz episode ...
1,a wonderful little production. the filming tec...,positive,wonderful little production. filming technique...
2,i thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,basically there's a family where a little boy ...,negative,basically there's family little boy (jake) thi...
4,"petter mattei's ""love in the time of money"" is...",positive,"petter mattei's ""love time money"" visually stu..."


In [64]:
x = df["clean_review"]
y = df["sentiment"]

In [65]:
x

,clean_review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. filming technique...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."
...,...
49995,thought movie right good job. creative origina...
49996,"bad plot, bad dialogue, bad acting, idiotic di..."
49997,catholic taught parochial elementary schools n...
49998,going disagree previous comment side maltin on...


In [66]:
y

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive
...,...
49995,positive
49996,negative
49997,negative
49998,negative


In [67]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [68]:
y

array([1, 1, 1, ..., 0, 0, 0])

In [69]:
from sklearn.model_selection import train_test_split
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.2 , shuffle=True)

In [70]:
x_train.shape , y_train.shape , x_test.shape , y_test.shape

((39665,), (39665,), (9917,), (9917,))

In [71]:
x_train

,clean_review
9197,"saw movie first berlin film festival, never se..."
20180,demented left-wing wipe-out trivializes dante'...
27577,"no, read stephen king novel ""thinner,"" choked ..."
7485,''the sentinel'' one best horror movies alread...
19549,"dvd watch, thinking would see type biography p..."
...,...
26848,prix de beauté made cusp changeover silence so...
1798,"""laugh, clown laugh"" released 1928, stars lege..."
2732,"ag excellent presentation drama, suspense thri..."
43071,"recently purchased lost horizon ebay, vivid me..."


In [72]:
import pandas as pd

x_test = pd.Series(x_test)
x_test = x_test.fillna('').astype(str)

In [73]:
df['clean_review'].apply(type).value_counts()

,count
clean_review,
<class 'str'>,49582


In [74]:
df['review'] = df['review'].fillna('').astype(str)

df['clean_review'] = df['review'].apply(remove_stopwords)

**BoW**

In [75]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000)
x_train_bow = cv.fit_transform(x_train)
x_test_bow = cv.transform(x_test)

In [76]:
x_train_bow.shape

(39665, 5000)

In [77]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(x_train_bow.toarray() , y_train)

GaussianNB()

In [78]:
y_pred = gnb.predict(x_test_bow.toarray())
from sklearn.metrics import accuracy_score , confusion_matrix
accuracy_score(y_test , y_pred)

0.7449833619038015

In [79]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(x_train_bow.toarray() , y_train)
y_pred = rf.predict(x_test_bow.toarray())
accuracy_score(y_test , y_pred)


0.8474336997075729

In [85]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, ngram_range=(1,2))

x_train_bow = cv.fit_transform(x_train)
x_test_bow = cv.transform(x_test)

# Better model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(x_train_bow, y_train)

from sklearn.metrics import accuracy_score

y_pred = model.predict(x_test_bow)
print(accuracy_score(y_test, y_pred))

0.8815165876777251


In [86]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [87]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(x_train_tfidf, y_train)

from sklearn.metrics import accuracy_score

y_pred = model.predict(x_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.8979530099828578


In [88]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()

model.fit(x_train_tfidf, y_train)

from sklearn.metrics import accuracy_score

y_pred = model.predict(x_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.862861752546133


**Word2Vec**

In [90]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.0 MB/s eta 0:00:00


In [91]:
from gensim.utils import simple_preprocess

X_train_tokens = [simple_preprocess(text) for text in x_train]
X_test_tokens  = [simple_preprocess(text) for text in x_test]

In [97]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=200,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # skip-gram
    sample=1e-5,
    epochs=10
)

In [98]:
import numpy as np

def sentence_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [99]:
X_train_w2v = np.array([sentence_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v  = np.array([sentence_vector(tokens, w2v_model) for tokens in X_test_tokens])

In [100]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_w2v, y_train)

LogisticRegression(max_iter=1000)

In [101]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_w2v)
print(accuracy_score(y_test, y_pred))

0.8811132398910961
